# 06 · Regularization 


In [2]:
# Add the project source folder to Python's search path.
# This allows us to import modules from src/museum_guide.
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../src").resolve()))

# Import the data preparation pipeline used by all project models.
# The function loads the images, creates the train/validation/test split,
# standardizes the data, and builds the PyTorch DataLoaders.
from museum_guide.data import prepare_subset

# Load the genre classification dataset.
# This uses the dataset generated in data/wikiart_genre_all.
genre = prepare_subset("genre")

# Basic checks to confirm the dataset was loaded correctly.
print("Loaded successfully")
print("Number of classes:", len(genre["classes"]))
print("Training batches:", len(genre["train_loader"]))

Loaded successfully
Number of classes: 11
Training batches: 37


In [4]:
# Build the same pooled MLP architecture selected in the FFN notebook.
# This is the baseline architecture (Try 7).

import torch
import torch.nn as nn

def make_mlp(n_classes, image_shape=(3, 32, 32), hidden_sizes=(256, 128)):
    layers = [nn.Flatten()]
    n_in = image_shape[0] * image_shape[1] * image_shape[2]

    for n_out in hidden_sizes:
        layers += [
            nn.Linear(n_in, n_out),
            nn.ReLU()
        ]
        n_in = n_out

    layers.append(nn.Linear(n_in, n_classes))

    model = nn.Sequential(*layers)

    for layer in model.modules():
        if isinstance(layer, nn.Linear):
            nn.init.kaiming_normal_(layer.weight, nonlinearity="relu")
            nn.init.zeros_(layer.bias)

    return model


def make_pooled_mlp(n_classes, pool_size=4):
    pooled_side = 128 // pool_size

    return nn.Sequential(
        nn.AvgPool2d(pool_size),
        make_mlp(
            n_classes=n_classes,
            image_shape=(3, pooled_side, pooled_side),
            hidden_sizes=(256, 128),
        ),
    )


model = make_pooled_mlp(len(genre["classes"]))

print(model)

Sequential(
  (0): AvgPool2d(kernel_size=4, stride=4, padding=0)
  (1): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=3072, out_features=256, bias=True)
    (2): ReLU()
    (3): Linear(in_features=256, out_features=128, bias=True)
    (4): ReLU()
    (5): Linear(in_features=128, out_features=11, bias=True)
  )
)


## Experiment 1: Weight Decay (L2 Regularization)

The feed-forward network notebooks were trained without explicit regularization.

In this experiment, we add L2 regularization through the `weight_decay`
parameter of the Adam optimizer and compare the results with the original model.

Objective:

- Reduce overfitting.
- Reduce the train-validation gap.
- Improve validation accuracy.

In [5]:
# Experiment 1: Weight Decay (L2 Regularization)

# Train the selected FFN architecture with weight decay.
# The only change compared to the original model is:
# weight_decay=1e-4

from museum_guide.training import fit

torch.manual_seed(0)

weight_decay_model = make_pooled_mlp(
    n_classes=len(genre["classes"]),
    pool_size=4,
)

loss_fn = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    weight_decay_model.parameters(),
    lr=1e-4,
    weight_decay=1e-4,  # L2 regularization
)

weight_decay_history = fit(
    weight_decay_model,
    genre["train_loader"],
    genre["val_loader"],
    loss_fn,
    optimizer,
    n_epochs=13,
)

print("Weight Decay experiment completed.")

epoch  1 | train loss 2.040 acc 0.296 | val loss 2.267 acc 0.222 | 0.5 s
epoch  2 | train loss 1.824 acc 0.385 | val loss 2.240 acc 0.216 | 0.3 s
epoch  3 | train loss 1.673 acc 0.460 | val loss 2.219 acc 0.228 | 0.3 s
epoch  4 | train loss 1.551 acc 0.522 | val loss 2.208 acc 0.246 | 0.3 s
epoch  5 | train loss 1.449 acc 0.574 | val loss 2.208 acc 0.240 | 0.3 s
epoch  6 | train loss 1.339 acc 0.623 | val loss 2.214 acc 0.238 | 0.3 s
epoch  7 | train loss 1.261 acc 0.651 | val loss 2.230 acc 0.260 | 0.4 s
epoch  8 | train loss 1.167 acc 0.706 | val loss 2.234 acc 0.246 | 0.7 s
epoch  9 | train loss 1.102 acc 0.732 | val loss 2.240 acc 0.252 | 0.7 s
epoch 10 | train loss 1.049 acc 0.754 | val loss 2.297 acc 0.236 | 0.7 s
epoch 11 | train loss 0.978 acc 0.779 | val loss 2.295 acc 0.238 | 0.6 s
epoch 12 | train loss 0.906 acc 0.806 | val loss 2.300 acc 0.240 | 0.7 s
epoch 13 | train loss 0.865 acc 0.820 | val loss 2.334 acc 0.258 | 0.6 s
Weight Decay experiment completed.


### Results

The feed-forward network was retrained with L2 regularization using:

- Adam optimizer
- Learning rate = 1e-4
- Weight decay = 1e-4

Observed result:

- Best validation accuracy: 26.0%

Comparison with the selected baseline model:

- Baseline validation accuracy: 26.6%
- Weight decay validation accuracy: 26.0%

### Conclusion

Weight decay did not improve validation performance on this dataset.

Although L2 regularization is theoretically expected to reduce overfitting, the selected feed-forward network already contains a strong dimensionality reduction step through average pooling.

The added regularization slightly reduced performance and is therefore not recommended for deployment based on the current experiments.

In [6]:
# -----------------------------------------------------------------------------
# Feed-Forward Network with Dropout Regularization
#
# This model is identical to the selected FFN architecture (Try 7),
# except that Dropout layers are added after each ReLU activation.
#
# Purpose:
# - Reduce overfitting
# - Prevent neurons from becoming overly specialized
# - Improve generalization on unseen paintings
# -----------------------------------------------------------------------------


def make_mlp_dropout(
    n_classes,
    image_shape=(3, 32, 32),
    hidden_sizes=(256, 128),
    dropout_rate=0.3,
):
    """
    Build a feed-forward neural network with Dropout.

    Args:
        n_classes: number of classes to predict.
        image_shape: shape of one input image after pooling.
        hidden_sizes: neurons in each hidden layer.
        dropout_rate: probability of dropping a neuron during training.

    Returns:
        A PyTorch Sequential model.
    """

    # Flatten the image:
    # (B, 3, 32, 32) -> (B, 3072)
    layers = [nn.Flatten()]

    n_in = image_shape[0] * image_shape[1] * image_shape[2]

    # Build each hidden layer
    for n_out in hidden_sizes:
        layers += [
            nn.Linear(n_in, n_out),     # Fully connected layer
            nn.ReLU(),                  # Non-linear activation
            nn.Dropout(dropout_rate),   # Dropout regularization
        ]
        n_in = n_out

    # Output layer (one logit per class)
    layers.append(nn.Linear(n_in, n_classes))

    model = nn.Sequential(*layers)

    # He initialization, recommended for ReLU networks
    for layer in model.modules():
        if isinstance(layer, nn.Linear):
            nn.init.kaiming_normal_(layer.weight, nonlinearity="relu")
            nn.init.zeros_(layer.bias)

    return model


def make_pooled_mlp_dropout(
    n_classes,
    pool_size=4,
    dropout_rate=0.3,
):
    """
    Build the complete dropout model.

    Architecture:
        AvgPool2d -> MLP with Dropout

    The average pooling layer reduces the image size before the
    fully connected network, lowering the number of parameters.
    """

    # Example:
    # 128 x 128 image -> 32 x 32 image with pool_size=4
    pooled_side = 128 // pool_size

    return nn.Sequential(
        nn.AvgPool2d(pool_size),

        make_mlp_dropout(
            n_classes=n_classes,
            image_shape=(3, pooled_side, pooled_side),
            hidden_sizes=(256, 128),
            dropout_rate=dropout_rate,
        ),
    )


# -------------------------------------------------------------------------
# Create the dropout model used for the experiment
# -------------------------------------------------------------------------

dropout_model = make_pooled_mlp_dropout(
    n_classes=len(genre["classes"]),
    pool_size=4,
    dropout_rate=0.3,
)

# Display the architecture to verify that Dropout layers were added
print(dropout_model)

Sequential(
  (0): AvgPool2d(kernel_size=4, stride=4, padding=0)
  (1): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=3072, out_features=256, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=256, out_features=128, bias=True)
    (5): ReLU()
    (6): Dropout(p=0.3, inplace=False)
    (7): Linear(in_features=128, out_features=11, bias=True)
  )
)


In [7]:
# -----------------------------------------------------------------------------
# Experiment 2: Dropout Regularization
#
# Train the same architecture as the selected FFN model, but with
# Dropout(p=0.3) added after each ReLU activation.
#
# The training setup remains unchanged:
# - Adam optimizer
# - Learning rate = 1e-4
# - 13 epochs
#
# This ensures a fair comparison with:
#   1. The original model
#   2. The Weight Decay experiment
# -----------------------------------------------------------------------------

from museum_guide.training import fit

# Fix the random seed so that the experiment is reproducible
torch.manual_seed(0)

# Create a fresh dropout model
dropout_model = make_pooled_mlp_dropout(
    n_classes=len(genre["classes"]),
    pool_size=4,
    dropout_rate=0.3,
)

# Same loss function as the original FFN notebook
loss_fn = nn.CrossEntropyLoss()

# Same optimizer and learning rate as the selected model
optimizer = torch.optim.Adam(
    dropout_model.parameters(),
    lr=1e-4,
)

# Train for the same number of epochs as the selected model
dropout_history = fit(
    dropout_model,
    genre["train_loader"],
    genre["val_loader"],
    loss_fn,
    optimizer,
    n_epochs=13,
)

print("Dropout experiment completed.")

epoch  1 | train loss 2.163 acc 0.251 | val loss 2.289 acc 0.202 | 0.5 s
epoch  2 | train loss 2.023 acc 0.291 | val loss 2.241 acc 0.222 | 0.4 s
epoch  3 | train loss 1.936 acc 0.335 | val loss 2.229 acc 0.212 | 0.4 s
epoch  4 | train loss 1.876 acc 0.366 | val loss 2.205 acc 0.248 | 0.4 s
epoch  5 | train loss 1.820 acc 0.403 | val loss 2.188 acc 0.254 | 0.4 s
epoch  6 | train loss 1.774 acc 0.429 | val loss 2.173 acc 0.244 | 0.3 s
epoch  7 | train loss 1.727 acc 0.447 | val loss 2.168 acc 0.260 | 0.4 s
epoch  8 | train loss 1.673 acc 0.484 | val loss 2.170 acc 0.254 | 0.4 s
epoch  9 | train loss 1.647 acc 0.500 | val loss 2.175 acc 0.268 | 0.4 s
epoch 10 | train loss 1.599 acc 0.515 | val loss 2.165 acc 0.268 | 0.3 s
epoch 11 | train loss 1.577 acc 0.534 | val loss 2.160 acc 0.268 | 0.3 s
epoch 12 | train loss 1.532 acc 0.549 | val loss 2.167 acc 0.244 | 0.4 s
epoch 13 | train loss 1.487 acc 0.573 | val loss 2.161 acc 0.256 | 0.3 s
Dropout experiment completed.


### Results

The dropout experiment used:

- Dropout rate = 0.3
- Learning rate = 1e-4
- 13 training epochs

Observed result:

- Best validation accuracy: 26.8%

Comparison:

- Baseline model: 26.6%
- Weight Decay: 26.0%
- Dropout: 26.8%

### Conclusion

Dropout achieved the best validation accuracy among the tested regularization strategies.

In addition, training accuracy remained substantially lower than in the baseline model, suggesting that the model relied less on memorization and generalized more effectively.

Based on these experiments, dropout appears to be the most promising regularization technique for the feed-forward network.

## Final Comparison

| Configuration | Best Validation Accuracy |
|---------------|-------------------------|
| Baseline | 26.6% |
| Weight Decay (1e-4) | 26.0% |
| Dropout (0.3) | 26.8% |

Recommendation: Dropout (0.3)

## Final Conclusion

Three regularization strategies were considered:

1. Weight Decay (L2 Regularization)
2. Dropout
3. Early Stopping

Results:

| Configuration | Best Validation Accuracy |
|---------------|-------------------------|
| Baseline | 26.6% |
| Weight Decay (1e-4) | 26.0% |
| Dropout (0.3) | 26.8% |

Weight decay did not improve validation performance relative to the baseline model.

Dropout achieved the highest validation accuracy among the tested regularization strategies. In addition, the model reached this performance while maintaining substantially lower training accuracy than the baseline configuration, suggesting improved generalization.

Early stopping remains an effective regularization technique and was already used in the selected feed-forward network by choosing the epoch with the highest validation accuracy.

### Recommendation

The recommended feed-forward network configuration is:

- Average Pooling (4×4)
- Hidden layers: 256 → 128
- Learning rate: 1e-4
- Dropout = 0.3
- Early Stopping

Based on the experiments, this configuration provides the best validation performance while limiting overfitting.

x